In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# Computing the Conjunctive Normal Form in First Order Logic

In order to convert a formula $f$ from first order logic into a set of clauses that is satisfiable if and only if $f$ is satisfiable,
we have to perform the following steps in order:
- eliminate biconditionals,
- eliminate conditionals,
- transform the formula into *negation normal form*,
  i.e. we push the negation symbol inwards,
- rename bound variables to avoid clashes,
- transform the formula into *prenex normal form*,
  i.e. we move the quantifiers outside,
- eliminate existential quantifiers by *skolemizing* the formula, and
- transform the formula into *clauses* in set notation.

When converting formulas into conjunctive normal form, we <u>assume</u> that the formulas are
*pure*, where we define a formula $f$ as *pure* if all quantifiers appearing in $f$ bind **different** variables.  For example, the formula
$$ \bigl(\forall x: p(x)\bigr) \vee \bigl(\forall x: q(x)\bigr)$$
is **not** *pure*, because there are two different universal quantifiers that both bind the same variable $x$.  We can rewrite this formulas as a *pure* formula by *renaming* all occurrences of $x$ that are bound by the second quantifier as follows:
$$ \bigl(\forall x: p(x)\bigr) \vee \bigl(\forall y: q(y)\bigr)$$

## Auxiliary Functions

Formulas are represented as values of the type `Formula` that is defined in the notebook `FOL-Parser.ipynb`, which also provides the function `parse` that converts a string into a formula.  The parser distinguishes variables and function symbols as follows:
- A word starting with a *lower* case letter is interpreted as a *variable*.
- A word starting with an *upper* case letter is assumed to be a *function* or *predicate symbol*.

In [2]:
importNotebook "FOL-Parser.ipynb"

imported 

FOL-Parser.ipynb

A formula in conjunctive normal form is represented in *set notation*, i.e. as a set of *clauses*, where every clause is a set of *literals*.  A literal is either an atomic formula or a negated atomic formula.  We define the following types:
- `Pos (p, [t1; ...; tn])` represents the literal $p(t_1, \cdots, t_n)$,
- `Neg (p, [t1; ...; tn])` represents the literal $\neg p(t_1, \cdots, t_n)$,
- a `Clause` is a set of literals.

These types are also used in the notebook `11-Prover.ipynb`.

In [3]:
type Literal =
    | Pos of string * Term list   // p(t1, ..., tn)
    | Neg of string * Term list   // ¬p(t1, ..., tn)

type Clause = Set<Literal>

The functions `literalToString`, `clauseToString`, and `clausesToString` convert literals, clauses, and sets of clauses into readable strings.  The function `termToString` is defined in the notebook `FOL-Parser.ipynb`.

In [4]:
let literalToString (l: Literal) : string =
    match l with
    | Pos (p, ts) -> termToString (Fun (p, ts))
    | Neg (p, ts) -> "¬" + termToString (Fun (p, ts))

let clauseToString (C: Clause) : string =
    "{" + String.concat ", " [ for l in C -> literalToString l ] + "}"

let clausesToString (Clauses: Set<Clause>) : string =
    "{\n" + String.concat ",\n" [ for C in Clauses -> "    " + clauseToString C ] + "\n}"

For testing purposes, the following formula is used.  This formula specifies the notion of a *grandparent*.

In [5]:
let s  = "∀g:∀c:(Grandparent(g, c) ↔ ∃p: (Parent(g, p) ∧ Parent(p, c)))"
let f1 = parse s
f1

Forall ("g", Forall ("c", Iff (Atom ("Grandparent", [Var "g"; Var "c"]), Exists ("p", And (Atom ("Parent", [Var "g"; Var "p"]), Atom ("Parent", [Var "p"; Var "c"]))))))

As the values of type `Formula` are hard to read, we will mostly display formulas using the function `formulaToString` from the notebook `FOL-Parser.ipynb`.

In [6]:
formulaToString f1

"∀g:∀c:(Grandparent(g, c) ↔ ∃p:(Parent(g, p) ∧ Parent(p, c)))"

A *substitution* $\sigma$ is represented as a map of the form $\{x_1 \mapsto s_1, \cdots, x_n \mapsto s_n\}$ that assigns terms to variables.

The function `applyTerm` takes two arguments:
- `σ` is a substitution,
- `t` is a term.

It returns the term $t\sigma$ that results from replacing every occurrence of a variable $x_i$ in `t` by the corresponding term $s_i$.

In [7]:
let rec applyTerm (σ: Map<string, Term>) (t: Term) : Term =
    match t with
    | Var x       -> match σ.TryFind x with
                     | Some s -> s
                     | None   -> t
    | Fun (f, ts) -> Fun (f, List.map (applyTerm σ) ts)

The function `applyFormula` takes two arguments:
- `σ` is a substitution,
- `f` is a formula.

It applies `σ` to all terms occurring in `f`.  If a variable $x$ that is bound by a quantifier is mapped to a variable $y$ by `σ`, then the quantifier is changed to bind $y$ instead of $x$.  This is used to *rename* bound variables.

In [8]:
let rec applyFormula (σ: Map<string, Term>) (f: Formula) : Formula =
    let rename (x: string) =
        match σ.TryFind x with
        | Some (Var y) -> y
        | _            -> x
    match f with
    | Verum | Falsum -> f
    | Atom (p, ts)   -> Atom (p, List.map (applyTerm σ) ts)
    | Not g          -> Not (applyFormula σ g)
    | And (g, h)     -> And (applyFormula σ g, applyFormula σ h)
    | Or  (g, h)     -> Or  (applyFormula σ g, applyFormula σ h)
    | Imp (g, h)     -> Imp (applyFormula σ g, applyFormula σ h)
    | Iff (g, h)     -> Iff (applyFormula σ g, applyFormula σ h)
    | Xor (g, h)     -> Xor (applyFormula σ g, applyFormula σ h)
    | Forall (x, g)  -> Forall (rename x, applyFormula σ g)
    | Exists (x, g)  -> Exists (rename x, applyFormula σ g)

In [9]:
f1

Forall ("g", Forall ("c", Iff (Atom ("Grandparent", [Var "g"; Var "c"]), Exists ("p", And (Atom ("Parent", [Var "g"; Var "p"]), Atom ("Parent", [Var "p"; Var "c"]))))))

In [10]:
formulaToString (applyFormula (Map [ ("g", Var "x"); ("p", Var "y"); ("c", Var "z") ]) f1)

"∀x:∀z:(Grandparent(x, z) ↔ ∃y:(Parent(x, y) ∧ Parent(y, z)))"

The function `boundVariables` takes one argument:
- `f` is a formula.

It returns the set of variables that are *bound* by a quantifier in `f`.

In [11]:
let rec boundVariables (f: Formula) : Set<string> =
    match f with
    | Forall (x, g) | Exists (x, g) -> (boundVariables g).Add x
    | Verum | Falsum | Atom _       -> Set.empty
    | Not g                         -> boundVariables g
    | And (g, h) | Or (g, h) | Imp (g, h) | Iff (g, h) | Xor (g, h)
                                    -> boundVariables g + boundVariables h

In [12]:
boundVariables f1

set ["c"; "g"; "p"]

The function `termVariables` takes one argument:
- `t` is a term.

It returns the set of all variables occurring in `t`.  The function `allVariables` takes a formula `f` and returns the set of all variables occurring in `f`.  This set contains both the variables occurring in terms and the variables bound by quantifiers.

In [13]:
let rec termVariables (t: Term) : Set<string> =
    match t with
    | Var x       -> set [ x ]
    | Fun (_, ts) -> Set.unionMany (List.map termVariables ts)

let rec allVariables (f: Formula) : Set<string> =
    match f with
    | Forall (x, g) | Exists (x, g) -> (allVariables g).Add x
    | Verum | Falsum                -> Set.empty
    | Atom (_, ts)                  -> Set.unionMany (List.map termVariables ts)
    | Not g                         -> allVariables g
    | And (g, h) | Or (g, h) | Imp (g, h) | Iff (g, h) | Xor (g, h)
                                    -> allVariables g + allVariables h

In [14]:
allVariables f1

set ["c"; "g"; "p"]

The function `renameBoundVariables` takes one argument:
- `f` is a formula.

It returns a formula that results from `f` by replacing all bound variables by **new** variables.  The new variables are single lower case letters that do not occur in `f`.  This only works if there are enough letters that do not already occur in $f$.  This approach would not be good enough for a production quality program,
but for the case of a demonstration it is sufficient.  The alternative would be to rename the variables as `x1`, `x2`, `x3`, $\cdots$, but that becomes unreadable very fast.

**Implementation:** `NewVars` is the sorted list of letters that do not occur in `f`.  `List.zip` combines the list of bound variables with the first elements of `NewVars` into a list of pairs, which is then turned into a substitution.

In [15]:
let renameBoundVariables (f: Formula) : Formula =
    let BoundVs = Set.toList (boundVariables f)
    let NewVars = [ for c in 'a' .. 'z' do
                      if not ((allVariables f).Contains (string c)) then string c ]
    let σ = Map (List.zip BoundVs (List.take BoundVs.Length NewVars |> List.map Var))
    applyFormula σ f

In [16]:
formulaToString (renameBoundVariables f1)

"∀b:∀a:(Grandparent(b, a) ↔ ∃d:(Parent(b, d) ∧ Parent(d, a)))"

## Eliminating Biconditionals

The function `eliminateBiconditional` takes one argument:
- `f` is a formula from first order logic.

It returns a formula that is equivalent to `f` and does not contain the operators '↔' and '⊕'.  This is done by using the following equivalences:
$$ (f \leftrightarrow g) \;\Leftrightarrow\; (f \rightarrow g) \wedge (g \rightarrow f) $$
$$ (f \oplus g) \;\Leftrightarrow\; (f \wedge \neg g) \vee (\neg f \wedge g) $$
In order to ensure that the resulting formula is <em style="color:blue">pure</em>, we have to rename the bound variables in the second copy of the formulas $f$ and $g$.

In [17]:
let rec eliminateBiconditional (f: Formula) : Formula =
    match f with
    | Iff (g, h) ->
        let ge = eliminateBiconditional g
        let he = eliminateBiconditional h
        And (Imp (ge, he), renameBoundVariables (Imp (he, ge)))
    | Xor (g, h) ->
        let ge = eliminateBiconditional g
        let he = eliminateBiconditional h
        Or (And (ge, Not he), renameBoundVariables (And (Not ge, he)))
    | Verum | Falsum | Atom _ -> f
    | Not g          -> Not (eliminateBiconditional g)
    | And (g, h)     -> And (eliminateBiconditional g, eliminateBiconditional h)
    | Or  (g, h)     -> Or  (eliminateBiconditional g, eliminateBiconditional h)
    | Imp (g, h)     -> Imp (eliminateBiconditional g, eliminateBiconditional h)
    | Forall (x, g)  -> Forall (x, eliminateBiconditional g)
    | Exists (x, g)  -> Exists (x, eliminateBiconditional g)

In [18]:
formulaToString f1

"∀g:∀c:(Grandparent(g, c) ↔ ∃p:(Parent(g, p) ∧ Parent(p, c)))"

In [19]:
let f2 = eliminateBiconditional f1
formulaToString f2

"∀g:∀c:((Grandparent(g, c) → ∃p:(Parent(g, p) ∧ Parent(p, c))) ∧ (∃a:(Parent(g, a) ∧ Parent(a, c)) → Grandparent(g, c)))"

## Eliminating Conditionals

The function `eliminateConditional` takes one argument:
- `f` is a formula from first order logic that does not contain the operators '↔' and '⊕'.

It returns a formula that is equivalent to `f` and does not contain the operator '→'.  This is done by using the following equivalence:
$$ (f \rightarrow g) \;\Leftrightarrow\; (\neg f \vee g) $$
The implementation of this function is similar to the implementation of the function `eliminateConditional` that we had used in propositional logic.  If `f` contains '↔' or '⊕', an exception is raised.

In [20]:
let rec eliminateConditional (f: Formula) : Formula =
    match f with
    | Imp (g, h)     -> Or (Not (eliminateConditional g), eliminateConditional h)
    | Verum | Falsum | Atom _ -> f
    | Not g          -> Not (eliminateConditional g)
    | And (g, h)     -> And (eliminateConditional g, eliminateConditional h)
    | Or  (g, h)     -> Or  (eliminateConditional g, eliminateConditional h)
    | Forall (x, g)  -> Forall (x, eliminateConditional g)
    | Exists (x, g)  -> Exists (x, eliminateConditional g)
    | Iff _ | Xor _  -> failwith "eliminateConditional: ↔ and ⊕ have to be eliminated first"

In [21]:
f2

Forall ("g", Forall ("c", And (Imp (Atom ("Grandparent", [Var "g"; Var "c"]), Exists ("p", And (Atom ("Parent", [Var "g"; Var "p"]), Atom ("Parent", [Var "p"; Var "c"])))), Imp (Exists ("a", And (Atom ("Parent", [Var "g"; Var "a"]), Atom ("Parent", [Var "a"; Var "c"]))), Atom ("Grandparent", [Var "g"; Var "c"])))))

In [22]:
let f3 = eliminateConditional f2
formulaToString f3

"∀g:∀c:((¬Grandparent(g, c) ∨ ∃p:(Parent(g, p) ∧ Parent(p, c))) ∧ (¬∃a:(Parent(g, a) ∧ Parent(a, c)) ∨ Grandparent(g, c)))"

## Negation Normal Form

The function `nnf` takes one argument:
- `f` is a formula that contains only the operators `¬`, `∧`, `∨`, `⊤`, `⊥`, and quantifiers.

It returns the <em style="color:blue;">negation normal form</em> of $f$, while `neg f` returns the *negation normal form* of $\neg f$.  The expression $\texttt{nnf}(f)$ is defined recursively as follows:
<ol>
    <li> $\texttt{nnf}(\neg \texttt{F}) = \texttt{neg}(\texttt{F})$, </li>
    <li> $\texttt{nnf}(\texttt{F}_1 \wedge \texttt{F}_2) =
          \texttt{nnf}(\texttt{F}_1) \wedge \texttt{nnf}(\texttt{F}_2)$,</li>
    <li> $\texttt{nnf}(\texttt{F}_1 \vee \texttt{F}_2) =
          \texttt{nnf}(\texttt{F}_1) \vee \texttt{nnf}(\texttt{F}_2)$.</li>
    <li> $\texttt{nnf}(\forall x: F) = \forall x: \texttt{nnf}(\texttt{F})$.</li>
    <li> $\texttt{nnf}(\exists x: F) = \exists x: \texttt{nnf}(\texttt{F})$.</li>
</ol>

The auxiliary function $\texttt{neg}$ is also defined recursively:
<ol>
    <li> $\texttt{neg}(p) = \texttt{nnf}(\neg p) = \neg p$ for all atomic formulas $p$,</li>
    <li> $\texttt{neg}(\neg F) = \texttt{nnf}(\neg \neg F) = \texttt{nnf}(F)$,</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \wedge F_2 \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \wedge F_2)\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \vee \neg F_2\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \vee \texttt{nnf}\bigl(\neg F_2\bigr) \\[0.1cm]
       = & \texttt{neg}(F_1) \vee \texttt{neg}(F_2).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \wedge F_2 \bigr) = \texttt{neg}(F_1) \vee \texttt{neg}(F_2)$.</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \vee F_2 \bigr)        \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \vee F_2) \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \wedge \neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \wedge \texttt{nnf}\bigl(\neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{neg}(F_1) \wedge \texttt{neg}(F_2).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \vee F_2 \bigr) = \texttt{neg}(F_1) \wedge \texttt{neg}(F_2)$.</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(\forall x: F \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg \forall x: F\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\exists x: \neg F\bigr) \\[0.1cm]
       = & \exists x: \texttt{nnf}(\neg F)           \\[0.1cm]
       = & \exists x: \texttt{neg}(F).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(\forall x: F \bigr) = \exists x: \texttt{neg}(F)$.</li>
      <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(\exists x: F \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg \exists x: F\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\forall x: \neg F\bigr) \\[0.1cm]
       = & \forall x: \texttt{nnf}(\neg F)           \\[0.1cm]
       = & \forall x: \texttt{neg}(F).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(\exists x: F \bigr) = \forall x: \texttt{neg}(F)$.</li>
</ol>

The functions `nnf` and `neg` are mutually recursive and are therefore defined together using `let rec ... and ...`.  If `f` contains one of the operators `→`, `↔`, or `⊕`, an exception is raised.

In [23]:
let rec nnf (f: Formula) : Formula =
    match f with
    | Verum | Falsum | Atom _ -> f
    | Not g          -> neg g
    | And (g, h)     -> And (nnf g, nnf h)
    | Or  (g, h)     -> Or  (nnf g, nnf h)
    | Forall (x, g)  -> Forall (x, nnf g)
    | Exists (x, g)  -> Exists (x, nnf g)
    | Imp _ | Iff _ | Xor _ -> failwith "nnf: →, ↔, and ⊕ have to be eliminated first"

and neg (f: Formula) : Formula =
    match f with
    | Verum          -> Falsum
    | Falsum         -> Verum
    | Atom _         -> Not f
    | Not g          -> nnf g
    | And (g, h)     -> Or  (neg g, neg h)
    | Or  (g, h)     -> And (neg g, neg h)
    | Forall (x, g)  -> Exists (x, neg g)
    | Exists (x, g)  -> Forall (x, neg g)
    | Imp _ | Iff _ | Xor _ -> failwith "neg: →, ↔, and ⊕ have to be eliminated first"

In [24]:
f3

Forall ("g", Forall ("c", And (Or (Not (Atom ("Grandparent", [Var "g"; Var "c"])), Exists ("p", And (Atom ("Parent", [Var "g"; Var "p"]), Atom ("Parent", [Var "p"; Var "c"])))), Or (Not (Exists ("a", And (Atom ("Parent", [Var "g"; Var "a"]), Atom ("Parent", [Var "a"; Var "c"])))), Atom ("Grandparent", [Var "g"; Var "c"])))))

In [25]:
let f4 = nnf f3
formulaToString f4

"∀g:∀c:((¬Grandparent(g, c) ∨ ∃p:(Parent(g, p) ∧ Parent(p, c))) ∧ (∀a:(¬Parent(g, a) ∨ ¬Parent(a, c)) ∨ Grandparent(g, c)))"

## Prenex Normal Form

In the following we assume that all quantifiers that occur in a formula bind **different** variables, i.e. we assume that the formulas are *pure*.  If this assumption is not satisfied, then the functions given below will produce <u>garbage</u>.

A *quantifier prefix* is a list of quantifiers together with the variables they bind, i.e. a list of the form
$$ [Q_1\,x_1; \cdots; Q_n\,x_n] $$
Here, the $Q_i$ denote quantifiers, i.e. we have $Q_i \in \{\forall, \exists\}$, while the $x_i$ are variables.  We represent an element $Q_i\,x_i$ of a quantifier prefix by a value of the type `Quantifier`.

In [26]:
type Quantifier =
    | All of string   // ∀x
    | Ex  of string   // ∃x

The function `mergeQuantifiers` takes two arguments:
- `Q1` is a quantifier prefix,
- `Q2` is a quantifier prefix.

It merges them into a new quantifier prefix such that the relative order of the quantifiers remains the same, i.e. if both $Q_1\,x_1$ and $Q_2\,x_2$ occur in `Q1` and $Q_1\,x_1$ occurs before $Q_2\,x_2$, then $Q_1\,x_1$ will occur before $Q_2\,x_2$ in the result.  Existential quantifiers are extracted first, since this results in Skolem functions with fewer arguments.

In [27]:
let rec mergeQuantifiers (Q1: Quantifier list) (Q2: Quantifier list) : Quantifier list =
    match Q1, Q2 with
    | [], _              -> Q2
    | _, []              -> Q1
    | (Ex x) :: rest, _  -> Ex x :: mergeQuantifiers rest Q2   // extract existential quantifiers first
    | _, (Ex y) :: rest  -> Ex y :: mergeQuantifiers Q1 rest
    | q :: rest, _       -> q :: mergeQuantifiers rest Q2

In [28]:
mergeQuantifiers [ All "x"; Ex "y" ] [ Ex "u"; All "v" ]

[Ex "u"; All "x"; Ex "y"; All "v"]

The function `extractQuantifiers` takes one argument:
- `f` is a pure formula in negation normal form.

It returns a pair `(Qs, m)`, where `Qs` is a quantifier prefix and `m` is the <em style="color:blue;">matrix</em> of the formula $f$.  The matrix of a formula is the part that remains when all quantifiers have been extracted.

In [29]:
let rec extractQuantifiers (f: Formula) : Quantifier list * Formula =
    match f with
    | And (g, h) ->
        let qg, gm = extractQuantifiers g
        let qh, hm = extractQuantifiers h
        // this works because f is pure
        mergeQuantifiers qg qh, And (gm, hm)
    | Or (g, h) ->
        let qg, gm = extractQuantifiers g
        let qh, hm = extractQuantifiers h
        mergeQuantifiers qg qh, Or (gm, hm)
    | Forall (x, g) ->
        let qg, gm = extractQuantifiers g
        All x :: qg, gm
    | Exists (x, g) ->
        let qg, gm = extractQuantifiers g
        Ex x :: qg, gm
    | _ ->   // f is ⊤, ⊥, a literal, or does not contain quantifiers
        [], f

In [30]:
f4

Forall ("g", Forall ("c", And (Or (Not (Atom ("Grandparent", [Var "g"; Var "c"])), Exists ("p", And (Atom ("Parent", [Var "g"; Var "p"]), Atom ("Parent", [Var "p"; Var "c"])))), Or (Forall ("a", Or (Not (Atom ("Parent", [Var "g"; Var "a"])), Not (Atom ("Parent", [Var "a"; Var "c"])))), Atom ("Grandparent", [Var "g"; Var "c"])))))

In [31]:
let qs, f5 = extractQuantifiers f4
qs, formulaToString f5

([All "g"; All "c"; Ex "p"; All "a"], "((¬Grandparent(g, c) ∨ (Parent(g, p) ∧ Parent(p, c))) ∧ ((¬Parent(g, a) ∨ ¬Parent(a, c)) ∨ Grandparent(g, c)))")

The function `attachQuantifiers` takes two arguments:
- `Qs` is a quantifier prefix,
- `m` is a formula, the matrix.

It combines the quantifiers `Qs` and the matrix `m` into a quantified formula.

In [32]:
let rec attachQuantifiers (Qs: Quantifier list) (m: Formula) : Formula =
    match Qs with
    | []           -> m
    | All x :: rest -> Forall (x, attachQuantifiers rest m)
    | Ex  x :: rest -> Exists (x, attachQuantifiers rest m)

In [33]:
let f6 = attachQuantifiers qs f5
formulaToString f6

"∀g:∀c:∃p:∀a:((¬Grandparent(g, c) ∨ (Parent(g, p) ∧ Parent(p, c))) ∧ ((¬Parent(g, a) ∨ ¬Parent(a, c)) ∨ Grandparent(g, c)))"

## Skolemization (Eliminating Existential Quantifiers)

The mutable variable `skolemCounter` is a global variable that is needed to create unique Skolem functions.  The function `skolemFunction` takes no arguments and returns a new name for a Skolem function every time it is called.

In [34]:
let mutable skolemCounter = 0

let skolemFunction () : string =
    skolemCounter <- skolemCounter + 1
    "sk" + string skolemCounter

The function `skolemize` takes two arguments:
- `f` is a formula in <em style="color:blue">prenex normal form</em>,
- `Vs` is a list of variables that are universally quantified.

It <em style="color:blue">skolemizes</em> the formula $f$, i.e. it replaces all existentially quantified variables by appropriate <em style="color:blue">Skolem functions</em>.  The arguments of these Skolem functions are the variables from `Vs`.

For skolemization to work correctly, we have to assume that
<font size="4" style="color:darkgreen; size:125%">$f$ does not contain free variables</font>!

In [35]:
let rec skolemize (f: Formula) (Vs: string list) : Formula =
    match f with
    | Exists (x, g) ->
        let t = Fun (skolemFunction (), List.map Var Vs)
        skolemize (applyFormula (Map [ (x, t) ]) g) Vs
    | Forall (x, g) ->
        skolemize g (Vs @ [ x ])
    | _ ->  // at this point f cannot contain a quantifier
        f

In [36]:
f6

Forall ("g", Forall ("c", Exists ("p", Forall ("a", And (Or (Not (Atom ("Grandparent", [Var "g"; Var "c"])), And (Atom ("Parent", [Var "g"; Var "p"]), Atom ("Parent", [Var "p"; Var "c"]))), Or (Or (Not (Atom ("Parent", [Var "g"; Var "a"])), Not (Atom ("Parent", [Var "a"; Var "c"]))), Atom ("Grandparent", [Var "g"; Var "c"])))))))

In [37]:
let f7 = skolemize f6 []
formulaToString f7

"((¬Grandparent(g, c) ∨ (Parent(g, sk1(g, c)) ∧ Parent(sk1(g, c), c))) ∧ ((¬Parent(g, a) ∨ ¬Parent(a, c)) ∨ Grandparent(g, c)))"

## Conversion to Clauses

The function `cnf` takes one argument:
- `f` is a <em style="color:blue">skolemized</em> formula from first order logic that is in <em style="color:blue">negation normal form</em> and does not contain quantifiers.

It returns the <em style="color:blue">conjunctive normal form</em> of $f$ in <em style="color:blue">set notation</em>.  This works the same way as in propositional logic.  If `f` does not have the required form, an exception is raised.

In [38]:
let rec cnf (f: Formula) : Set<Clause> =
    match f with
    | Verum               -> Set.empty
    | Falsum              -> set [ Set.empty ]
    | Atom (p, ts)        -> set [ set [ Pos (p, ts) ] ]
    | Not (Atom (p, ts))  -> set [ set [ Neg (p, ts) ] ]
    | And (g, h)          -> cnf g + cnf h
    | Or  (g, h)          -> set [ for k1 in cnf g do for k2 in cnf h -> k1 + k2 ]
    | _                   -> failwith "cnf: formula has the wrong form"

In [39]:
f7

And (Or (Not (Atom ("Grandparent", [Var "g"; Var "c"])), And (Atom ("Parent", [Var "g"; Fun ("sk1", [Var "g"; Var "c"])]), Atom ("Parent", [Fun ("sk1", [Var "g"; Var "c"]); Var "c"]))), Or (Or (Not (Atom ("Parent", [Var "g"; Var "a"])), Not (Atom ("Parent", [Var "a"; Var "c"]))), Atom ("Grandparent", [Var "g"; Var "c"])))

In [40]:
let f8 = cnf f7
printfn "%s" (clausesToString f8)

{
    {Grandparent(g, c), ¬Parent(a, c), ¬Parent(g, a)},
    {Parent(g, sk1(g, c)), ¬Grandparent(g, c)},
    {Parent(sk1(g, c), c), ¬Grandparent(g, c)}
}

## Putting Everything Together

The function `normalize` takes one argument:
- `f` is a <em style="color:blue">pure</em> formula from first order logic that **must not** contain free variables.

It transforms $f$ into a set of first order clauses that is satisfiable if and only if `f` is satisfiable.

In [41]:
let normalize (f: Formula) : Set<Clause> =
    let f1     = eliminateBiconditional f
    let f2     = eliminateConditional f1
    let f3     = nnf f2
    let qs, f4 = extractQuantifiers f3
    let f5     = attachQuantifiers qs f4
    let f6     = skolemize f5 []
    cnf f6

In [42]:
normalize f1

set [set [Pos ("Grandparent", [Var "g"; Var "c"]); Neg ("Parent", [Var "a"; Var "c"]); Neg ("Parent", [Var "g"; Var "a"])]; set [Pos ("Parent", [Var "g"; Fun ("sk2", [Var "g"; Var "c"])]); Neg ("Grandparent", [Var "g"; Var "c"])]; set [Pos ("Parent", [Fun ("sk2", [Var "g"; Var "c"]); Var "c"]); Neg ("Grandparent", [Var "g"; Var "c"])]]

The function `test` takes one argument:
- `s` is a string representing a pure formula without free variables.

It computes the clauses of this formula and prints them.

In [43]:
let test (s: string) : unit =
    let f = parse s
    printfn "The CNF of %s is:" s
    printfn "%s" (clausesToString (normalize f))

In [44]:
test s

The CNF of 

∀g:∀c:(Grandparent(g, c) ↔ ∃p: (Parent(g, p) ∧ Parent(p, c)))

 is:

{
    {Grandparent(g, c), ¬Parent(a, c), ¬Parent(g, a)},
    {Parent(g, sk3(g, c)), ¬Grandparent(g, c)},
    {Parent(sk3(g, c), c), ¬Grandparent(g, c)}
}

In [45]:
test "¬(∃y:∀x:P(x,y)→∀u:∃v:P(u,v))"

The CNF of 

¬(∃y:∀x:P(x,y)→∀u:∃v:P(u,v))

 is:

{
    {P(x, sk4())},
    {¬P(sk5(), v)}
}